[Guide Index](README.md)


# 03c SPARQL Entailment

`query()` can take an `entailment=` argument, which applies an *entailment regime* to a graph before running the query, without changing the graph. It accepts `ENTAILMENT.RDF`, `ENTAILMENT.RDFS`, `ENTAILMENT['OWL-RDF-Based']` and `ENTAILMENT['OWL-Direct']`. The entailment regimes are explained in the [Graph Inferencing guide](02e-graphs-inferencing.ipynb).

This guide covers query-time entailment.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier sections.


In [1]:
from starlayer.graph import ENTAILMENT, NamedNode
from starlayer.graph import StarLayerGraph

EX = "http://example.org/"
RDF_TYPE = NamedNode("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")
OWL_RL = ENTAILMENT['OWL-RDF-Based']

## 1. Entailment regimes

Pass the regime to `query()`; the graph itself is never modified.

- `ENTAILMENT.RDF` and `ENTAILMENT.RDFS`: the query is rewritten so that Oxigraph answers it as if the entailed triples were present.
- `ENTAILMENT['OWL-RDF-Based']` (OWL 2 RL, through `owlrl`) and `ENTAILMENT['OWL-Direct']` (OWL 2 DL, through HermiT or `rustdl`): the store is copied, the entailed triples are added to the copy, and the query runs on the copy. The copy is kept between queries (see section 2).

In the example below, a Manager is a subclass of Employee, and Manager is the same class as Supervisor (`owl:equivalentClass`). RDFS has a rule for subclasses but none for equivalent classes, so only the OWL regime finds the Supervisors.

In [2]:
g = StarLayerGraph(prefixes={"ex": EX})
# alice is a manager. With entailment she is also an employee, and a supervisor
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:Manager owl:equivalentClass ex:Supervisor .
    ex:alice a ex:Manager .
""", format="turtle")

QUERY = "SELECT ?x WHERE { ?x a ex:Employee }"
SUPERVISORS = "SELECT ?x WHERE { ?x a ex:Supervisor }"

print("Employees (Manager is a subclass of Employee, and alice is a Manager):")
print("  no entailment:   ", [x.x.value for x in g.query(QUERY)])
print("  RDFS:            ", [x.x.value for x in g.query(QUERY, entailment=ENTAILMENT.RDFS)])
print("  OWL RL:          ", [x.x.value for x in g.query(QUERY, entailment=ENTAILMENT['OWL-RDF-Based'])])
print("  RDF:             ", [x.x.value for x in g.query(QUERY, entailment=ENTAILMENT.RDF)], "(RDF entailment has no subclass rule)")

print("\nSupervisors (Manager is the same class as Supervisor):")
print("  no entailment:   ", [x.x.value for x in g.query(SUPERVISORS)])
print("  RDFS:            ", [x.x.value for x in g.query(SUPERVISORS, entailment=ENTAILMENT.RDFS)], "(RDFS has no equivalent-class rule)")
print("  OWL RL:          ", [x.x.value for x in g.query(SUPERVISORS, entailment=ENTAILMENT['OWL-RDF-Based'])])
print("  RDF:             ", [x.x.value for x in g.query(SUPERVISORS, entailment=ENTAILMENT.RDF)])

print("\nThe graph is unchanged:", len(g), "triples")

Employees (Manager is a subclass of Employee, and alice is a Manager):
  no entailment:    []
  RDFS:             ['http://example.org/alice']
  OWL RL:           ['http://example.org/alice']
  RDF:              [] (RDF entailment has no subclass rule)

Supervisors (Manager is the same class as Supervisor):
  no entailment:    []
  RDFS:             [] (RDFS has no equivalent-class rule)
  OWL RL:           ['http://example.org/alice']
  RDF:              []

The graph is unchanged: 3 triples


## 2. Caching the entailed triples: `recompute=`

Computing the OWL entailments can take significant time depending on the size and complexity of the graph. Queries using `ENTAILMENT['OWL-RDF-Based']` and `ENTAILMENT['OWL-Direct']` copy the store and add the entailed triples to the copy. The copy is kept between queries, so a repeated query costs only the query. The `recompute=` parameter controls when the graph is copied and the entailments are computed again.

- `recompute="onChange"` (the default): makes the copy again, and computes the entailments again, only if the underlying graph has changed.
- `recompute="never"`: uses the kept copy whatever has changed since. Use it when you do not need the answers to reflect changes made to the graph between queries. A copy is made only if none is kept.
- `recompute="always"`: makes the copy and computes the entailments on every call, and keeps none. The kept copy is dropped. (Frees memory.)

`recompute=` has no effect on `RDF` and `RDFS`.

When you have finished querying with an OWL regime, call `clear_entailment_cache()` to drop the copy, which holds the whole graph in memory.

The following code demonstrates the impact of caching and query-time entailment.

In [3]:
import time

# Build a graph of some size: the people, the chain of classes and the class that ends it.
def build_graph(people=300, depth=8):
    lines = [
        "@prefix ex: <http://example.org/> .",
        "@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .",
        "@prefix owl: <http://www.w3.org/2002/07/owl#> .",
    ]
    lines += [f"ex:Level{i} rdfs:subClassOf ex:Level{i + 1} ." for i in range(depth)]
    lines.append(f"ex:Level{depth} owl:equivalentClass ex:High .")
    lines += [f"ex:person{i} a ex:Level0 ; ex:knows ex:person{(i + 1) % people} ." for i in range(people)]
    graph = StarLayerGraph(prefixes={"ex": EX})
    graph.parse(data="\n".join(lines), format="turtle")
    return graph

g_big = build_graph()
print(len(g_big), "triples")

# Run the same query and say how long it took. The options are passed to query().
COUNT_HIGH = "SELECT (COUNT(?x) AS ?n) WHERE { ?x a ex:High }"

def timed(label, **options):
    start = time.perf_counter()
    count = list(g_big.query(COUNT_HIGH, entailment=OWL_RL, **options))[0].n.value
    print(f"{label:<62} {time.perf_counter() - start:5.2f} seconds ({count} people are a High)")

timed("first query: computes the entailed triples and keeps them")
timed("again: onChange reuses the kept triples, nothing changed", recompute="onChange")
timed('recompute="always": computes them again, and keeps none', recompute="always")
timed("next call: nothing was kept, so they are computed (and kept)")

# Clear the kept copy and query again: nothing is kept, so the copy is made again.
g_big.clear_entailment_cache()
timed("after clearing: nothing was kept, so they are computed again")

# The querying is done: drop the copy, which holds the whole graph in memory.
g_big.clear_entailment_cache()


609 triples


first query: computes the entailed triples and keeps them       0.58 seconds (300 people are a High)
again: onChange reuses the kept triples, nothing changed        0.00 seconds (300 people are a High)


recompute="always": computes them again, and keeps none         0.60 seconds (300 people are a High)


next call: nothing was kept, so they are computed (and kept)    0.59 seconds (300 people are a High)


after clearing: nothing was kept, so they are computed again    0.60 seconds (300 people are a High)


<StarLayerGraph DEFAULT (609 triples)>

## 3. Detecting inconsistencies

An ontology is *inconsistent* when its statements cannot all be true at the same time. For example, `ex:Dog` is declared disjoint with `ex:Cat`, so nothing can be both, and `ex:fido` is asserted to be both. A reasoner cannot draw useful conclusions from a graph containing a contradiction, so it reports it. The Inferencing guide's [section 3.5](02e-graphs-inferencing.ipynb#3.5-Detecting-inconsistencies) explains this topic in more detail.

`ENTAILMENT['OWL-RDF-Based']` reports an inconsistency as data, not as an exception: the entailed triples hold an `err:ErrorMessage` with an `err:error` message, and the query returns a result. A simple `ASK` detects an inconsistency and a `SELECT` reads the message. `ENTAILMENT['OWL-Direct']` raises `InconsistentOntologyError` instead.

In [4]:
# A consistent graph, for comparison: alice is a manager, and manager and teamlead are the same class
g_owl = StarLayerGraph(prefixes={"ex": EX})
g_owl.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
""", format="turtle")

# An inconsistent graph: Dog and Cat are declared disjoint, but fido is asserted to be both
g_bad = StarLayerGraph(prefixes={"ex": EX})
g_bad.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Dog owl:disjointWith ex:Cat .
    ex:fido a ex:Dog, ex:Cat .
''', format="turtle")

# The entailed triples hold an err:ErrorMessage when the graph is inconsistent. ASK finds one, SELECT reads its message.
ASK_INCONSISTENT = """
PREFIX err: <http://www.daml.org/2002/03/agents/agent-ont#>
ASK { ?e a err:ErrorMessage }
"""

print("Is the graph inconsistent? (ASK)")
print("  inconsistent graph, with OWL RL:   ", g_bad.query(ASK_INCONSISTENT, entailment=OWL_RL).askAnswer)
print("  inconsistent graph, no entailment: ", g_bad.query(ASK_INCONSISTENT).askAnswer)
print("  consistent graph, with OWL RL:     ", g_owl.query(ASK_INCONSISTENT, entailment=OWL_RL).askAnswer)

ERROR_MESSAGES = """
PREFIX err: <http://www.daml.org/2002/03/agents/agent-ont#>
SELECT ?message WHERE { ?e err:error ?message }
"""

print("\nWhy? (the message of the error)")
for row in g_bad.query(ERROR_MESSAGES, entailment=OWL_RL):
    print("  ", row.message.value)

# A normal query still returns a result: nothing is retracted, and the inconsistency is not shown in it.
rows = g_bad.query("SELECT ?c WHERE { ex:fido a ?c }", entailment=OWL_RL)
print("\nfido's classes (still returned despite the inconsistency):", sorted(r.c.value.rsplit("/", 1)[1] for r in rows))


Is the graph inconsistent? (ASK)
  inconsistent graph, with OWL RL:    True
  inconsistent graph, no entailment:  False
  consistent graph, with OWL RL:      False

Why? (the message of the error)
   Disjoint classes http://example.org/Dog and http://example.org/Cat have a common individual http://example.org/fido

fido's classes (still returned despite the inconsistency): ['Cat', 'Dog']


## 4. What is rejected

`entailment=` takes only the four regimes above; any other value (such as `ENTAILMENT.D`, which is a recognized regime IRI but not implemented) raises `NotImplementedError` rather than returning an answer that ignores the regime. `engine=` and `timeout=` choose the reasoner (`"hermit"` or `"rustdl"`) and limit its time, and apply only to `ENTAILMENT['OWL-Direct']`; with anything else they raise `ValueError`. The OWL-Direct reasoners are described in the [DL reasoning guide](appendix-b2-dl-reasoning-examples.ipynb).

In [5]:
for label, call in [
    ("entailment=D:", lambda: g.query(QUERY, entailment=ENTAILMENT.D)),
    ("engine= without OWL-Direct:", lambda: g.query(QUERY, entailment=ENTAILMENT.RDFS, engine="rustdl")),
]:
    try:
        call()
    except (NotImplementedError, ValueError) as error:
        print(f"{label:30s}", type(error).__name__ + ":", str(error)[:110])

entailment=D:                  NotImplementedError: entailment must be one of ENTAILMENT.RDF, RDFS, ['OWL-RDF-Based'], ['OWL-Direct'], got <NamedNode value=http:/
engine= without OWL-Direct:    ValueError: engine='rustdl' is only meaningful for entailment=ENTAILMENT['OWL-Direct'] (got entailment=<NamedNode value=ht
